In [0]:
# ---------------------------------------------------------
# 0. Run Imports 
# ---------------------------------------------------------

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable


In [0]:
# ---------------------------------------------------------
# 1. Extract Schema From DataFrame
# ---------------------------------------------------------
def extract_schema_dict(df):

    schema_dict = {}

    for field in df.schema.fields:

        schema_dict[field.name] = str(field.dataType)

    return schema_dict

In [0]:
# ---------------------------------------------------------
# 2. Get Schema From An Existing Table
# ---------------------------------------------------------

def get_existing_table_schema(
    spark,
    bronze_table
):

    if not spark.catalog.tableExists(bronze_table):

        return {}

    table_df = spark.table(bronze_table)

    return extract_schema_dict(table_df)

In [0]:
# ---------------------------------------------------------
# 3. Compare Schema From An Existing Table VS DataFrame
# ---------------------------------------------------------

def compare_schemas(
    incoming_schema,
    existing_schema
):

    schema_changes = []

    # -----------------------------------
    # NEW COLUMNS
    # -----------------------------------

    for column in incoming_schema:

        if column not in existing_schema:

            schema_changes.append({
                "change_type": "NEW_COLUMN",
                "column_name": column,
                "old_data_type": None,
                "new_data_type": incoming_schema[column]
            })

    # -----------------------------------
    # REMOVED COLUMNS
    # -----------------------------------

    for column in existing_schema:

        if column not in incoming_schema:

            schema_changes.append({
                "change_type": "REMOVED_COLUMN",
                "column_name": column,
                "old_data_type": existing_schema[column],
                "new_data_type": None
            })

    # -----------------------------------
    # DATA TYPE CHANGES
    # -----------------------------------

    for column in incoming_schema:

        if column in existing_schema:

            if incoming_schema[column] != existing_schema[column]:

                schema_changes.append({
                    "change_type": "DATA_TYPE_CHANGE",
                    "column_name": column,
                    "old_data_type": existing_schema[column],
                    "new_data_type": incoming_schema[column]
                })

    return schema_changes

In [0]:
# ---------------------------------------------------------
# 4. Write Schema Changes To Schema Drift Table
# ---------------------------------------------------------

def write_schema_changes(
    spark,
    schema_changes,
    schema_log_table,
    source_system,
    bronze_table,
    batch_id,
    pipeline_run_id,
    file_path
):

    if len(schema_changes) == 0:

        return

    rows = []

    for change in schema_changes:

        rows.append((
            source_system,
            bronze_table,
            batch_id,
            pipeline_run_id,
            file_path,
            change["change_type"],
            change["column_name"],
            change["old_data_type"] if change["old_data_type"] is not None else "",
            change["new_data_type"] if change["new_data_type"] is not None else ""
        ))

    # Define explicit schema to avoid type inference issues
    schema = StructType([
        StructField("source_system", StringType(), True),
        StructField("bronze_table", StringType(), True),
        StructField("batch_id", StringType(), True),
        StructField("pipeline_run_id", StringType(), True),
        StructField("file_path", StringType(), True),
        StructField("change_type", StringType(), True),
        StructField("column_name", StringType(), True),
        StructField("old_data_type", StringType(), True),
        StructField("new_data_type", StringType(), True)
    ])

    schema_df = spark.createDataFrame(
        rows,
        schema
    ).withColumn(
        "detected_timestamp",
        F.current_timestamp()
    )

    (
        schema_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(schema_log_table)
    )